# 06_baseline: v2モデル（TimeSeriesSplit対応版）の動作確認

全モデルのv2版（通常版 + Optuna版）を動作確認します。
- 特徴量エンジニアリングなし（処理時間短縮のため）
- Optunaのn_trialsは最小限（3回）
- cv_strategy="stratified"でテスト

## ライブラリインストール

In [0]:
%pip install lightgbm xgboost catboost scikit-learn pytorch-tabnet torch --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

## 共通設定

In [0]:
%load_ext autoreload
%autoreload 2

import datetime
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

# プロジェクトルート設定
PROJECT_ROOT = Path("/Workspace/Users/1122hkaito@gmail.com/jaggle_2026")
sys.path.insert(0, str(PROJECT_ROOT))

# 共通パラメータ
SEED = 42
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# データパス
DATA_DIR = PROJECT_ROOT / "data" / "input"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / datetime.datetime.now().strftime("%Y%m%d")
SAVED_MODELS_DIR = OUTPUT_DIR / "models"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"TARGET_COL: {TARGET_COL}")
print(f"ID_COL: {ID_COL}")

PROJECT_ROOT: /Workspace/Users/1122hkaito@gmail.com/jaggle_2026
DATA_DIR: /Workspace/Users/1122hkaito@gmail.com/jaggle_2026/data/input
TARGET_COL: 10年定着ラベル
ID_COL: 社員ID


## データ読み込み

In [0]:
# データ読み込み
persona_train = pd.read_csv(DATA_DIR / "employee_persona_train.csv")
persona_test = pd.read_csv(DATA_DIR / "employee_persona_test.csv")
monthly_train = pd.read_csv(DATA_DIR / "employee_monthly_train.csv")
monthly_test = pd.read_csv(DATA_DIR / "employee_monthly_test.csv")

print(f"persona_train shape: {persona_train.shape}")
print(f"persona_test shape: {persona_test.shape}")
print(f"monthly_train shape: {monthly_train.shape}")
print(f"monthly_test shape: {monthly_test.shape}")

# 月次データを社員IDごとに集約（簡易版：特徴量エンジニアリングなし）
def aggregate_monthly(df):
    """monthlyデータを社員IDごとに集約"""
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 集約対象外のカラムを除外
    exclude_cols = [ID_COL, "経過月数"]
    agg_cols = [col for col in num_cols if col not in exclude_cols]
    
    # mean, max, minを計算
    agg_df = df.groupby(ID_COL)[agg_cols].agg(['mean', 'max', 'min']).reset_index()
    agg_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in agg_df.columns]
    
    return agg_df

monthly_train_agg = aggregate_monthly(monthly_train)
monthly_test_agg = aggregate_monthly(monthly_test)

# personaとmonthlyを結合
train = persona_train.merge(monthly_train_agg, on=ID_COL, how="left")
test = persona_test.merge(monthly_test_agg, on=ID_COL, how="left")

print(f"\n結合後 Train shape: {train.shape}")
print(f"結合後 Test shape: {test.shape}")

persona_train shape: (2761, 20)
persona_test shape: (2502, 19)
monthly_train shape: (65754, 29)
monthly_test shape: (60048, 29)

結合後 Train shape: (2761, 74)
結合後 Test shape: (2502, 73)


## 特徴量エンジニアリング（簡易版）

In [0]:
def build_features(train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str):
    """特徴量エンジニアリングを行う関数（簡易版：処理時間短縮のため最小限）"""
    train_proc = train.copy()
    test_proc = test.copy()

    # 非数値列（文字列・日付列）を一時除外
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    
    # 社員ID は後で使うため除外対象から外す
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
        
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")

    # ターゲットとIDを分離
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    test_ids = test_proc[id_col]
    
    # NaN値を0で埋める
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    return X_train, y_train, X_test, test_ids

# 特徴量作成
X_train, y_train, X_test, test_ids = build_features(train, test, TARGET_COL, ID_COL)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train value counts:")
print(y_train.value_counts())

X_train shape: (2761, 57)
y_train shape: (2761,)
X_test shape: (2502, 57)

y_train value counts:
10年定着ラベル
1    1559
0    1202
Name: count, dtype: int64


## 基本パラメータ設定

In [0]:
# 全モデル共通のベースパラメータ
base_params = {
    "n_splits": 3,  # 処理時間短縮のため3分割
    "seed": SEED,
    "save_dir": None,
    "cv_strategy": "stratified",  # stratifiedでテスト
}

# Optuna用パラメータ（最小限）
optuna_base_params = base_params.copy()
optuna_base_params["n_trials"] = 3  # 最小限のトライアル数

print("Base params:", base_params)
print("Optuna base params:", optuna_base_params)

Base params: {'n_splits': 3, 'seed': 42, 'save_dir': None, 'cv_strategy': 'stratified'}
Optuna base params: {'n_splits': 3, 'seed': 42, 'save_dir': None, 'cv_strategy': 'stratified', 'n_trials': 3}


In [0]:
# 結果を格納する辞書
results = {}
optuna_results = {}

## LightGBM v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - LightGBM v2 学習開始")
print("=" * 60)

from common.lgbm.lgbm_model_v2 import run_lgb

lgbm_params = base_params.copy()
lgbm_params.update({
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "early_stopping_rounds": 50,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_lgb(data, lgbm_params)
oof_score = result_data["oof_score"]

print(f"\nLightGBM v2 OOF Score (Log Loss): {oof_score:.6f}")
results["LightGBM"] = oof_score

2026-08-05 15:25:50.309574 - LightGBM v2 学習開始

LightGBM v2 OOF Score (Log Loss): 0.601802


## XGBoost v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - XGBoost v2 学習開始")
print("=" * 60)

from common.xgboost.xgb_model_v2 import run_xgb

xgboost_params = base_params.copy()
xgboost_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "learning_rate": 0.05,
    "max_depth": 6,
    "early_stopping_rounds": 50,
    "verbose_eval": False,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_xgb(data, xgboost_params)
oof_score = result_data["oof_score"]

print(f"\nXGBoost v2 OOF Score (Log Loss): {oof_score:.6f}")
results["XGBoost"] = oof_score

2026-08-05 15:26:13.709053 - XGBoost v2 学習開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:26:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:26:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:26:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()



XGBoost v2 OOF Score (Log Loss): 0.632985


## CatBoost v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - CatBoost v2 学習開始")
print("=" * 60)

from common.catboost.cat_model_v2 import run_cat

catboost_params = base_params.copy()
catboost_params.update({
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": 100,
    "learning_rate": 0.05,
    "early_stopping_rounds": 50,
    "verbose": False,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_cat(data, catboost_params)
oof_score = result_data["oof_score"]

print(f"\nCatBoost v2 OOF Score (Log Loss): {oof_score:.6f}")
results["CatBoost"] = oof_score

2026-08-05 15:28:07.988280 - CatBoost v2 学習開始

CatBoost v2 OOF Score (Log Loss): 0.593425


## HistGradientBoosting v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - HistGradientBoosting v2 学習開始")
print("=" * 60)

from common.histgb.histgb_model_v2 import run_histgb

histgb_params = base_params.copy()
histgb_params.update({
    "learning_rate": 0.05,
    "max_iter": 100,
    "max_depth": 6,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_histgb(data, histgb_params)
oof_score = result_data["oof_score"]

print(f"\nHistGradientBoosting v2 OOF Score (Log Loss): {oof_score:.6f}")
results["HistGradientBoosting"] = oof_score

2026-08-05 15:28:14.487436 - HistGradientBoosting v2 学習開始

HistGradientBoosting v2 OOF Score (Log Loss): 0.609782


## Logistic Regression v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Logistic Regression v2 学習開始")
print("=" * 60)

from common.logistic.logistic_model_v2 import run_logistic

logistic_params = base_params.copy()
logistic_params.update({
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_logistic(data, logistic_params)
oof_score = result_data["oof_score"]

print(f"\nLogistic Regression v2 OOF Score (Log Loss): {oof_score:.6f}")
results["Logistic Regression"] = oof_score

2026-08-05 15:28:23.492384 - Logistic Regression v2 学習開始

Logistic Regression v2 OOF Score (Log Loss): 0.592532


## Neural Network v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Neural Network v2 学習開始")
print("=" * 60)

from common.nn.nn_model_v2 import run_nn

nn_params = base_params.copy()
nn_params.update({
    "hidden_dim": 64,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 128,
    "epochs": 30,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_nn(data, nn_params)
oof_score = result_data["oof_score"]

print(f"\nNeural Network v2 OOF Score (Log Loss): {oof_score:.6f}")
results["Neural Network"] = oof_score

2026-08-05 15:28:40.156249 - Neural Network v2 学習開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten



Neural Network v2 OOF Score (Log Loss): 0.615564


## TabNet v2 テスト

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - TabNet v2 学習開始")
print("=" * 60)

# モジュールを再読み込み
import importlib
import sys
if 'common.tabnet.tabnet_model_v2' in sys.modules:
    importlib.reload(sys.modules['common.tabnet.tabnet_model_v2'])

from common.tabnet.tabnet_model_v2 import run_tabnet

tabnet_params = base_params.copy()
tabnet_params.update({
    "n_d": 32,
    "n_a": 32,
    "n_steps": 3,
    "gamma": 1.3,
    "lambda_sparse": 1e-3,
    "max_epochs": 50,
    "patience": 10,
    "batch_size": 256,
    "virtual_batch_size": 128,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

result_data, _ = run_tabnet(data, tabnet_params)
oof_score = result_data["oof_score"]

print(f"\nTabNet v2 OOF Score (Log Loss): {oof_score:.6f}")
results["TabNet"] = oof_score

2026-08-05 15:35:09.157082 - TabNet v2 学習開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.85492 | val_0_logloss: 4.6609  |  0:00:00s
epoch 1  | loss: 0.73433 | val_0_logloss: 6.85117 |  0:00:01s
epoch 2  | loss: 0.68522 | val_0_logloss: 2.30424 |  0:00:02s
epoch 3  | loss: 0.64308 | val_0_logloss: 1.8579  |  0:00:03s
epoch 4  | loss: 0.63049 | val_0_logloss: 0.93146 |  0:00:04s
epoch 5  | loss: 0.61862 | val_0_logloss: 1.86883 |  0:00:05s
epoch 6  | loss: 0.59255 | val_0_logloss: 1.25604 |  0:00:06s
epoch 7  | loss: 0.59752 | val_0_logloss: 0.98158 |  0:00:07s
epoch 8  | loss: 0.58708 | val_0_logloss: 1.47586 |  0:00:08s
epoch 9  | loss: 0.595   | val_0_logloss: 1.36532 |  0:00:09s
epoch 10 | loss: 0.58774 | val_0_logloss: 1.0253  |  0:00:09s
epoch 11 | loss: 0.60008 | val_0_logloss: 1.15582 |  0:00:10s
epoch 12 | loss: 0.60513 | val_0_logloss: 1.15258 |  0:00:11s
epoch 13 | loss: 0.58808 | val_0_logloss: 0.87899 |  0:00:12s
epoch 14 | loss: 0.58302 | val_0_logloss: 0.7731  |  0:00:12s
epoch 15 | loss: 0.57094 | val_0_logloss: 0.72181 |  0:00:13s
epoch 16

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.99828 | val_0_logloss: 3.41343 |  0:00:00s
epoch 1  | loss: 0.78017 | val_0_logloss: 7.22197 |  0:00:01s
epoch 2  | loss: 0.70948 | val_0_logloss: 1.40498 |  0:00:02s
epoch 3  | loss: 0.67336 | val_0_logloss: 2.32196 |  0:00:03s
epoch 4  | loss: 0.68228 | val_0_logloss: 0.98374 |  0:00:04s
epoch 5  | loss: 0.67001 | val_0_logloss: 0.82733 |  0:00:05s
epoch 6  | loss: 0.63845 | val_0_logloss: 1.2051  |  0:00:06s
epoch 7  | loss: 0.63278 | val_0_logloss: 0.95206 |  0:00:07s
epoch 8  | loss: 0.60902 | val_0_logloss: 0.92583 |  0:00:08s
epoch 9  | loss: 0.60597 | val_0_logloss: 0.89491 |  0:00:09s
epoch 10 | loss: 0.61351 | val_0_logloss: 0.83593 |  0:00:09s
epoch 11 | loss: 0.61651 | val_0_logloss: 0.83099 |  0:00:10s
epoch 12 | loss: 0.60967 | val_0_logloss: 0.83236 |  0:00:11s
epoch 13 | loss: 0.59406 | val_0_logloss: 0.80888 |  0:00:12s
epoch 14 | loss: 0.5787  | val_0_logloss: 0.78485 |  0:00:13s
epoch 15 | loss: 0.58051 | val_0_logloss: 0.8074  |  0:00:15s
epoch 16

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.94992 | val_0_logloss: 2.84649 |  0:00:00s
epoch 1  | loss: 0.71031 | val_0_logloss: 4.377   |  0:00:01s
epoch 2  | loss: 0.66306 | val_0_logloss: 3.9477  |  0:00:02s
epoch 3  | loss: 0.62495 | val_0_logloss: 1.36688 |  0:00:03s
epoch 4  | loss: 0.59225 | val_0_logloss: 1.86061 |  0:00:04s
epoch 5  | loss: 0.60258 | val_0_logloss: 1.21039 |  0:00:06s
epoch 6  | loss: 0.58743 | val_0_logloss: 1.23571 |  0:00:07s
epoch 7  | loss: 0.57997 | val_0_logloss: 0.99716 |  0:00:08s
epoch 8  | loss: 0.59456 | val_0_logloss: 1.79204 |  0:00:09s
epoch 9  | loss: 0.58078 | val_0_logloss: 1.00288 |  0:00:10s
epoch 10 | loss: 0.57899 | val_0_logloss: 1.17087 |  0:00:10s
epoch 11 | loss: 0.56712 | val_0_logloss: 0.95074 |  0:00:11s
epoch 12 | loss: 0.57061 | val_0_logloss: 1.04763 |  0:00:12s
epoch 13 | loss: 0.56644 | val_0_logloss: 0.86038 |  0:00:12s
epoch 14 | loss: 0.56916 | val_0_logloss: 0.75291 |  0:00:14s
epoch 15 | loss: 0.55368 | val_0_logloss: 0.75488 |  0:00:15s
epoch 16

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



TabNet v2 OOF Score (Log Loss): 0.664608


In [0]:
# モジュールを再読み込み
import importlib
import sys
if 'common.tabnet.tabnet_model_v2' in sys.modules:
    importlib.reload(sys.modules['common.tabnet.tabnet_model_v2'])

## 通常版結果まとめ

In [0]:
# 結果を表示
results_df = pd.DataFrame(list(results.items()), columns=["Model", "OOF Score"])
results_df = results_df.sort_values("OOF Score")
print("\n" + "=" * 60)
print("通常版モデル結果まとめ")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)


通常版モデル結果まとめ
               Model  OOF Score
 Logistic Regression   0.592532
            CatBoost   0.593425
            LightGBM   0.601802
HistGradientBoosting   0.609782
      Neural Network   0.615564
             XGBoost   0.632985
              TabNet   0.664608


## Optuna版テスト（ハイパーパラメータ最適化）

In [0]:
%pip install optuna --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### LightGBM Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - LightGBM Optuna v2 最適化開始")
print("=" * 60)

from common.lgbm.lgbm_model_optuna_v2 import run_lgb_optuna

lgbm_optuna_params = optuna_base_params.copy()
lgbm_optuna_params.update({
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "early_stopping_rounds": 50,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_lgb_optuna(data, lgbm_optuna_params)
best_score = result_data["best_score"]

print(f"\nLightGBM Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["LightGBM"] = best_score

2026-08-05 15:44:51.460851 - LightGBM Optuna v2 最適化開始

LightGBM Optuna v2 Best Score (Log Loss): 0.590201
Best Params: {'learning_rate': 0.06054365855469246, 'num_leaves': 95, 'max_depth': 3, 'min_child_samples': 98, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}


### XGBoost Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - XGBoost Optuna v2 最適化開始")
print("=" * 60)

from common.xgboost.xgb_model_optuna_v2 import run_xgb_optuna

xgboost_optuna_params = optuna_base_params.copy()
xgboost_optuna_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "early_stopping_rounds": 50,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_xgb_optuna(data, xgboost_optuna_params)
best_score = result_data["best_score"]

print(f"\nXGBoost Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["XGBoost"] = best_score

2026-08-05 15:45:49.263896 - XGBoost Optuna v2 最適化開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:45:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:45:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:45:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf


XGBoost Optuna v2 Best Score (Log Loss): 0.605655
Best Params: {'learning_rate': 0.13394334706750485, 'max_depth': 7, 'min_child_weight': 15, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'alpha': 0.31044435499483225, 'lambda': 8.148018307012941e-07}


### CatBoost Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - CatBoost Optuna v2 最適化開始")
print("=" * 60)

from common.catboost.cat_model_optuna_v2 import run_cat_optuna

catboost_optuna_params = optuna_base_params.copy()
catboost_optuna_params.update({
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": 100,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_cat_optuna(data, catboost_optuna_params)
best_score = result_data["best_score"]

print(f"\nCatBoost Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["CatBoost"] = best_score

2026-08-05 15:46:06.312423 - CatBoost Optuna v2 最適化開始

CatBoost Optuna v2 Best Score (Log Loss): 0.599816
Best Params: {'learning_rate': 0.015957084694148364, 'depth': 3, 'l2_leaf_reg': 2.9154431891537547, 'random_strength': 0.001026006512489678, 'bagging_temperature': 0.7080725777960455}


### HistGradientBoosting Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - HistGradientBoosting Optuna v2 最適化開始")
print("=" * 60)

from common.histgb.histgb_model_optuna_v2 import run_histgb_optuna

histgb_optuna_params = optuna_base_params.copy()
histgb_optuna_params.update({
    # HistGBはOptunaで全パラメータを探索
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_histgb_optuna(data, histgb_optuna_params)
best_score = result_data["best_score"]

print(f"\nHistGradientBoosting Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["HistGradientBoosting"] = best_score

2026-08-05 15:46:47.496718 - HistGradientBoosting Optuna v2 最適化開始

HistGradientBoosting Optuna v2 Best Score (Log Loss): 0.604890
Best Params: {'learning_rate': 0.012184186502221764, 'max_iter': 447, 'max_depth': 10, 'min_samples_leaf': 72, 'l2_regularization': 1.5320059381854043e-08, 'max_bins': 249}


### Logistic Regression Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Logistic Regression Optuna v2 最適化開始")
print("=" * 60)

from common.logistic.logistic_model_optuna_v2 import run_logistic_optuna

logistic_optuna_params = optuna_base_params.copy()
logistic_optuna_params.update({
    "max_iter": 1000,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_logistic_optuna(data, logistic_optuna_params)
best_score = result_data["best_score"]

print(f"\nLogistic Regression Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["Logistic Regression"] = best_score

2026-08-05 15:48:10.619320 - Logistic Regression Optuna v2 最適化開始


/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Logistic Regression Optuna v2 Best Score (Log Loss): 0.590042
Best Params: {'C': 0.39079671568228835, 'penalty': 'l1'}


### Neural Network Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - Neural Network Optuna v2 最適化開始")
print("=" * 60)

from common.nn.nn_model_optuna_v2 import run_nn_optuna

nn_optuna_params = optuna_base_params.copy()
nn_optuna_params.update({
    # NNはOptunaで全パラメータを探索
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_nn_optuna(data, nn_optuna_params)
best_score = result_data["best_score"]

print(f"\nNeural Network Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["Neural Network"] = best_score

2026-08-05 15:48:30.118534 - Neural Network Optuna v2 最適化開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten



Neural Network Optuna v2 Best Score (Log Loss): 0.648470
Best Params: {'hidden_dim': 116, 'dropout': 0.4802857225639665, 'learning_rate': 0.0029106359131330704, 'batch_size': 64, 'epochs': 34}


### TabNet Optuna v2

In [0]:
print("=" * 60)
print(f"{datetime.datetime.now()} - TabNet Optuna v2 最適化開始")
print("=" * 60)

from common.tabnet.tabnet_model_optuna_v2 import run_tabnet_optuna

tabnet_optuna_params = optuna_base_params.copy()
tabnet_optuna_params.update({
    "batch_size": 256,
    "virtual_batch_size": 128,
})

data = {
    "X_train": X_train,
    "y_train": y_train,
}

result_data, best_params = run_tabnet_optuna(data, tabnet_optuna_params)
best_score = result_data["best_score"]

print(f"\nTabNet Optuna v2 Best Score (Log Loss): {best_score:.6f}")
print(f"Best Params: {result_data['best_params']}")
optuna_results["TabNet"] = best_score

2026-08-05 15:50:21.952242 - TabNet Optuna v2 最適化開始


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.4476  | val_0_logloss: 4.38043 |  0:00:01s
epoch 1  | loss: 1.00637 | val_0_logloss: 5.44383 |  0:00:03s
epoch 2  | loss: 0.95214 | val_0_logloss: 5.44469 |  0:00:06s
epoch 3  | loss: 0.8803  | val_0_logloss: 4.97921 |  0:00:09s
epoch 4  | loss: 0.90939 | val_0_logloss: 2.81295 |  0:00:10s
epoch 5  | loss: 0.86192 | val_0_logloss: 2.65325 |  0:00:12s
epoch 6  | loss: 0.95694 | val_0_logloss: 7.89336 |  0:00:15s
epoch 7  | loss: 0.93394 | val_0_logloss: 2.08482 |  0:00:17s
epoch 8  | loss: 0.76834 | val_0_logloss: 2.79175 |  0:00:19s
epoch 9  | loss: 0.78184 | val_0_logloss: 3.05781 |  0:00:21s
epoch 10 | loss: 0.67163 | val_0_logloss: 1.47376 |  0:00:23s
epoch 11 | loss: 0.65493 | val_0_logloss: 1.27014 |  0:00:25s
epoch 12 | loss: 0.63696 | val_0_logloss: 0.90342 |  0:00:27s
epoch 13 | loss: 0.62873 | val_0_logloss: 0.77275 |  0:00:29s
epoch 14 | loss: 0.63456 | val_0_logloss: 0.74229 |  0:00:31s
epoch 15 | loss: 0.63012 | val_0_logloss: 0.77425 |  0:00:33s
epoch 16

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.39544 | val_0_logloss: 6.3651  |  0:00:02s
epoch 1  | loss: 1.03076 | val_0_logloss: 6.73148 |  0:00:04s
epoch 2  | loss: 0.89719 | val_0_logloss: 4.57049 |  0:00:06s
epoch 3  | loss: 0.82775 | val_0_logloss: 2.75097 |  0:00:08s
epoch 4  | loss: 0.87442 | val_0_logloss: 2.75501 |  0:00:11s
epoch 5  | loss: 0.76193 | val_0_logloss: 2.07863 |  0:00:13s
epoch 6  | loss: 0.72064 | val_0_logloss: 1.16457 |  0:00:15s
epoch 7  | loss: 0.69473 | val_0_logloss: 1.18347 |  0:00:17s
epoch 8  | loss: 0.67899 | val_0_logloss: 1.25157 |  0:00:19s
epoch 9  | loss: 0.66898 | val_0_logloss: 1.06173 |  0:00:21s
epoch 10 | loss: 0.66466 | val_0_logloss: 1.63612 |  0:00:24s
epoch 11 | loss: 0.68723 | val_0_logloss: 1.14968 |  0:00:26s
epoch 12 | loss: 0.6522  | val_0_logloss: 1.21955 |  0:00:28s
epoch 13 | loss: 0.68946 | val_0_logloss: 0.99045 |  0:00:30s
epoch 14 | loss: 0.73997 | val_0_logloss: 0.76959 |  0:00:33s
epoch 15 | loss: 0.65201 | val_0_logloss: 0.7201  |  0:00:35s
epoch 16

...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.99434 | val_0_logloss: 5.33175 |  0:00:01s
epoch 1  | loss: 0.81845 | val_0_logloss: 5.0989  |  0:00:02s
epoch 2  | loss: 0.74253 | val_0_logloss: 1.85914 |  0:00:03s
epoch 3  | loss: 0.69374 | val_0_logloss: 1.04759 |  0:00:04s
epoch 4  | loss: 0.67098 | val_0_logloss: 1.52628 |  0:00:05s
epoch 5  | loss: 0.64106 | val_0_logloss: 1.08851 |  0:00:07s
epoch 6  | loss: 0.63236 | val_0_logloss: 0.99471 |  0:00:08s
epoch 7  | loss: 0.61587 | val_0_logloss: 1.18934 |  0:00:10s
epoch 8  | loss: 0.62619 | val_0_logloss: 0.87578 |  0:00:11s
epoch 9  | loss: 0.6214  | val_0_logloss: 1.19044 |  0:00:12s
epoch 10 | loss: 0.62125 | val_0_logloss: 0.89655 |  0:00:13s
epoch 11 | loss: 0.61764 | val_0_logloss: 0.89446 |  0:00:15s
epoch 12 | loss: 0.60059 | val_0_logloss: 0.92235 |  0:00:16s
epoch 13 | loss: 0.59658 | val_0_logloss: 0.8527  |  0:00:17s
epoch 14 | loss: 0.60314 | val_0_logloss: 0.79444 |  0:00:19s
epoch 15 | loss: 0.60129 | val_0_logloss: 0.89343 |  0:00:20s
epoch 16

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 2.01458 | val_0_logloss: 3.48804 |  0:00:01s
epoch 1  | loss: 0.99583 | val_0_logloss: 3.62079 |  0:00:02s
epoch 2  | loss: 0.765   | val_0_logloss: 2.34641 |  0:00:03s
epoch 3  | loss: 0.70649 | val_0_logloss: 2.42642 |  0:00:05s
epoch 4  | loss: 0.67358 | val_0_logloss: 1.21299 |  0:00:06s
epoch 5  | loss: 0.66279 | val_0_logloss: 0.77086 |  0:00:08s
epoch 6  | loss: 0.65629 | val_0_logloss: 0.97859 |  0:00:09s
epoch 7  | loss: 0.65222 | val_0_logloss: 0.7765  |  0:00:10s
epoch 8  | loss: 0.65493 | val_0_logloss: 0.75852 |  0:00:11s
epoch 9  | loss: 0.64669 | val_0_logloss: 0.77943 |  0:00:12s
epoch 10 | loss: 0.63544 | val_0_logloss: 0.85639 |  0:00:14s
epoch 11 | loss: 0.64028 | val_0_logloss: 0.96088 |  0:00:16s
epoch 12 | loss: 0.63967 | val_0_logloss: 0.76515 |  0:00:17s
epoch 13 | loss: 0.62721 | val_0_logloss: 0.75283 |  0:00:19s
epoch 14 | loss: 0.63218 | val_0_logloss: 0.72075 |  0:00:21s
epoch 15 | loss: 0.62674 | val_0_logloss: 0.69606 |  0:00:22s
epoch 16

[W 2026-08-05 16:05:22,248] Trial 2 failed with parameters: {'n_d': 18, 'n_a': 18, 'n_steps': 5, 'gamma': 1.524756431632238, 'lambda_sparse': 1.9762189340280066e-05, 'max_epochs': 93, 'patience': 22} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Workspace/Users/1122hkaito@gmail.com/jaggle_2026/common/tabnet/tabnet_model_optuna_v2.py", line 118, in objective_func
    model.fit(
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py", line 258, in fit
    self._train_epoch(train_dataloader)
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-a4627b29-b7be-4319-bf18-b09b8e209119/lib/python3.12/site-packages/pytorc

## Optuna版結果まとめ

In [0]:
# Optuna結果を表示
optuna_results_df = pd.DataFrame(list(optuna_results.items()), columns=["Model", "Best Score"])
optuna_results_df = optuna_results_df.sort_values("Best Score")
print("\n" + "=" * 60)
print("Optuna版モデル結果まとめ")
print("=" * 60)
print(optuna_results_df.to_string(index=False))
print("=" * 60)


Optuna版モデル結果まとめ
               Model  Best Score
 Logistic Regression    0.590042
            LightGBM    0.590201
            CatBoost    0.599816
HistGradientBoosting    0.604890
             XGBoost    0.605655
      Neural Network    0.648470


## 全体結果比較

In [0]:
# 通常版とOptuna版を並べて比較
comparison_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Normal OOF Score": list(results.values()),
    "Optuna Best Score": [optuna_results.get(m, None) for m in results.keys()],
})
comparison_df["Improvement"] = comparison_df["Normal OOF Score"] - comparison_df["Optuna Best Score"]
comparison_df = comparison_df.sort_values("Optuna Best Score")

print("\n" + "=" * 80)
print("通常版 vs Optuna版 比較")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)
print("\n✅ 全v2モデルの動作確認が完了しました！")


通常版 vs Optuna版 比較
Empty DataFrame
Columns: [Model, Normal OOF Score, Optuna Best Score, Improvement]
Index: []

✅ 全v2モデルの動作確認が完了しました！
